# TP2 - Etapa 2: entrenamiento, evaluación y comparación de modelos

Notebook de entrenamiento y estudio de resultados: evaluación de la búsqueda por similitud
(Etapa 1, NDCG@10), fine-tuning de ResNet (Modelo A), CNN propia (Modelo B) y comparación entre
modelos (ver `ENUNCIADO.md`, secciones 4.3 y 5). **Ambos modelos son obligatorios.**

El entrenamiento se realiza en Colab para aprovechar la GPU:
**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

Requisitos previos (en tu fork):
- Funciones de la Etapa 1 implementadas en `src/lib/services/similarity_service.py`.
- `train_classifier`, `evaluate_classifier` y `extract_custom_embedding` implementadas en
  `src/lib/services/classifier_service.py` (esta notebook solo las orquesta).
- Splits y conjunto independiente generados en `1-pre-processing.ipynb`.

Al finalizar:
- Descargar los checkpoints generados y colocarlos en `models/` del entorno local
  (la aplicación los usa en las pestañas de las Etapas 1, 2 y 3).
- Publicarlos en un link público de solo lectura para los docentes.
- Entregar esta notebook ejecutada, con sus salidas.

**Notebook en Google Colab (si corresponde):** <reemplazar por el link, con permiso de lectura para los docentes>

Si entregan la notebook en Colab, esta copia del repositorio funciona como notebook de referencia:
el link de arriba es obligatorio y la notebook de Colab debe estar ejecutada, con sus salidas
(ver `ENUNCIADO.md`, sección 8.2).

## Equipo
- Alumno 1 :
- Alumno 2 :

## 1. Entorno

La notebook corre en Colab (GPU T4) o localmente desde la raíz del repositorio. En Colab:

- Se monta Google Drive y ahí se guardan **checkpoints, historiales y resultados de experimentos**
  (`MyDrive/tp2-perros/`): si Colab se desconecta no se pierde nada, y al volver a ejecutar las celdas
  los experimentos ya terminados no se repiten.
- El dataset se copia al disco local de la máquina (`/content`), no se lee desde Drive: leer miles de
  JPG chicos desde Drive hace que la GPU espere al disco.

Si el fork es privado, usar un token de acceso en `REPO_URL` (no dejarlo escrito en la notebook entregada).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lucasramire/tuia-dog-recognition-app.git"
BRANCH = "feature/valentin-lujambio"  # pasar a main cuando este integrado

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path("/content/proyecto").exists():
        !git clone --branch $BRANCH $REPO_URL /content/proyecto
    %cd /content/proyecto
    !git log --oneline -1
    !pip install -q pydantic-settings python-dotenv kagglehub albumentations onnx onnxruntime ultralytics imagehash
print("Colab:", IN_COLAB)

## 2. Configuración

Las rutas se fijan como variables de entorno **antes** de importar `lib` (igual que en
`1-pre-processing.ipynb`). Sin pgvector: la evaluación usa matrices de numpy.

In [ ]:
import json
import logging
import os
import shutil
import time
from dataclasses import asdict

ROOT = Path.cwd()
assert (ROOT / "src" / "lib").is_dir(), "Ejecutar la notebook desde la raiz del repositorio"
PERSIST = Path("/content/drive/MyDrive/tp2-perros") if IN_COLAB else ROOT
os.environ.update(
    {
        "USE_PGVECTOR": "false",
        "DATA_PATH": str(ROOT / "data"),
        "DATASET_PATH": str(ROOT / "data" / "dataset"),
        "EMBEDDINGS_PATH": str(ROOT / "data" / "embeddings.json"),
        "OUTPUT_PATH": str(PERSIST / "output"),
        "MODEL_PATH": str(PERSIST / "models"),
        "TORCH_HOME": str(ROOT / "models" / "torch"),
        "YOLO_MODEL": str(ROOT / "models" / "yolov8n.pt"),
    }
)
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from lib.config import settings

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("lib.training").setLevel(logging.INFO)  # una linea por epoca
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 140)

DATASET = settings.dataset_path
EXP_DIR = settings.output_path / "experiments"
for d in (settings.model_path, EXP_DIR):
    d.mkdir(parents=True, exist_ok=True)
SEED = 42
NUM_WORKERS = min(4, os.cpu_count() or 1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("dataset:", DATASET)
print("checkpoints:", settings.model_path, "| experimentos:", EXP_DIR)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no", "| workers:", NUM_WORKERS)

## 3. Dataset y splits

El dataset se descarga con `scripts/download_dataset.py` (kagglehub; si pide credenciales, cargar
`KAGGLE_USERNAME` y `KAGGLE_KEY` en los *secrets* de Colab). En Colab se guarda además un zip en Drive
para no volver a descargarlo.

Los splits son los de `1-pre-processing.ipynb`: `data/splits/splits_v1.csv` (sin fugas entre splits ni
duplicados; `valid` y `test` con 10 imágenes por raza). Todo lo que sigue usa **solo `train` y `valid`
para elegir**; `test` se mira una vez por modelo final.

In [ ]:
ZIP = PERSIST / "dataset.zip"
if not (DATASET / "dogs.csv").exists():
    if IN_COLAB and ZIP.exists():
        shutil.unpack_archive(str(ZIP), str(DATASET))
    else:
        !python scripts/download_dataset.py
        if IN_COLAB:
            shutil.make_archive(str(ZIP.with_suffix("")), "zip", str(DATASET))
# Conjunto independiente (imagenes fuera de git, D-29): en Colab, desde un zip en Drive.
INDEPENDENT_ZIP = PERSIST / "independent.zip"
if IN_COLAB and INDEPENDENT_ZIP.exists() and not (DATASET / "independent").exists():
    shutil.unpack_archive(str(INDEPENDENT_ZIP), str(DATASET))
print(sorted(p.name for p in DATASET.iterdir() if not p.name.startswith(".")))

In [ ]:
from lib.dataset.manifest import class_names, load_manifest

MANIFEST = settings.data_path / "splits" / "splits_v1.csv"
manifest = load_manifest(MANIFEST, dataset_path=DATASET, check_files=True)
CLASSES = class_names(manifest)
counts = manifest.groupby(["breed", "split"]).size().unstack(fill_value=0)[["train", "valid", "test"]]
display(counts.agg(["sum", "min", "median", "max"]).T)
print(f"{len(CLASSES)} razas | train por raza: {counts['train'].min()} a {counts['train'].max()} "
      f"({counts['train'].max() / counts['train'].min():.1f} veces)")

## 5. Etapa 1 — Evaluación de la búsqueda por similitud (baseline)

TO-DO:
- Galería: imágenes de `train`. Consultas: `test` y el conjunto independiente (nunca indexadas).
- Para cada consulta, recuperar las Top-10 y calcular NDCG@10 con relevancia binaria (misma raza).
- Reportar NDCG@10 medio, precision@10 y accuracy de la raza predicha, por separado para `test` y
  para el conjunto independiente.
- Comparar `cosine` vs `l2`, analizar `TOP_K` y calibrar `SIMILARITY_THRESHOLD` con validación.
- Justificar el resultado con ejemplos de consultas y sus Top-10.

Sugerencia: no hace falta la base de datos. Calculá los embeddings de la galería en una matriz de
`numpy` y buscá con un producto matricial (`EmbeddingStore.append` reescribe el JSON en cada llamada).

In [ ]:
from lib.bootstrap import build_similarity, build_store
from lib.evaluation.metrics import ndcg_at_k

similarity = build_similarity(settings, build_store(settings))

## TO-DO: embeddings de la galería con similarity.extract_embedding, búsqueda de las consultas,
## NDCG@10 (ndcg_at_k con n_relevant = imágenes de la raza en la galería), precision@10 y accuracy;
## comparación cosine vs l2 y calibración del umbral.

## Experimentos: cómo se corren

Cada variante se entrena con `lib.training.trainer.train` (la misma función que usa `train_classifier`)
y guarda su checkpoint e historial en `output/experiments/`. Receta común (plan, D-23): AdamW,
scheduler coseno, label smoothing 0,1, weight decay 1e-4, batch 64, precisión mixta en GPU, early stopping
con paciencia 5 sobre la accuracy de `valid`, semilla 42. Se guarda la **mejor época según `valid`**, no la
última.

La selección (estrategia, hiperparámetros) se hace con `valid`. Una vez elegida la variante, el modelo
final se entrena con `classifier.train_classifier(config)`, que escribe el checkpoint que usa la app.

Presupuesto estimado en una T4: 15 a 25 s por época; unas 5 a 6 horas de época en total como máximo,
en la práctica menos por el early stopping.

In [ ]:
from lib.training.trainer import TrainConfig, train
from lib.visualization.training import plot_confusion, plot_histories


def base_config(**overrides) -> TrainConfig:
    return TrainConfig(image_size=settings.image_size, num_workers=NUM_WORKERS, seed=SEED, **overrides)


def run_experiment(name: str, **overrides) -> dict:
    """Entrena una variante; si ya termino antes (archivo .done.json en Drive), reutiliza el resultado."""
    done = EXP_DIR / f"{name}.done.json"
    if done.exists():
        return json.loads(done.read_text())
    start = time.perf_counter()
    result = train(base_config(**overrides), manifest, DATASET, EXP_DIR / f"{name}.pth", EXP_DIR / f"{name}_history.json")
    history = load_history(name)
    summary = {
        "name": name,
        "best_val_accuracy": result.best_val_accuracy,
        "best_epoch": result.best_epoch,
        "epochs_run": result.epochs_run,
        "minutes": (time.perf_counter() - start) / 60,
        "trainable_parameters": history["trainable_parameters"],
        "overrides": overrides,
    }
    done.write_text(json.dumps(summary, indent=2))
    return summary


def load_history(name: str) -> dict:
    return json.loads((EXP_DIR / f"{name}_history.json").read_text())


def results_table(results: list[dict]) -> pd.DataFrame:
    table = pd.DataFrame(results).set_index("name")
    table["minutes"] = table["minutes"].round(1)
    return table[["best_val_accuracy", "best_epoch", "epochs_run", "minutes", "trainable_parameters"]].round(4)

## 6. Modelo A (obligatorio): fine-tuning de ResNet18

**Arquitectura.** ResNet18 preentrenada en ImageNet (`IMAGENET1K_V1`), con la capa `fc` reemplazada por
una cabeza lineal de 512 a 70 razas. El embedding para la búsqueda por similitud es la salida del
*global average pooling* (512 dimensiones), justo antes de la cabeza.

**Estrategias que se comparan** (transfer learning, plan D-21):

| Variante | Qué se entrena | LR | Idea |
|---|---|---|---|
| A1 feature extraction | solo la cabeza (backbone y sus BatchNorm congelados) | cabeza 1e-3 | ImageNet ya tiene ~118 razas de perro: las features genéricas deberían alcanzar bastante |
| A2 fine-tuning completo | todo | cabeza 1e-3, backbone 1e-4 | adapta también las capas profundas; LR menor en el backbone para no borrar lo preentrenado (olvido catastrófico) |
| A3 mejor de A1/A2 + pesos de clase | igual que la mejor | igual | `train` tiene de 65 a 198 imágenes por raza; pesos inversos a la frecuencia en la loss (D-25) |

En feature extraction los BatchNorm del backbone quedan en modo evaluación: si se dejaran en modo
entrenamiento cambiarían sus estadísticas aunque sus pesos estén congelados.

In [ ]:
A_COMMON = dict(model="resnet18_finetuned")
results_a = [
    run_experiment("A1_feature_extraction", **A_COMMON, strategy="feature_extraction"),
    run_experiment("A2_finetune", **A_COMMON, strategy="finetune"),
]
best_strategy = max(results_a, key=lambda r: r["best_val_accuracy"])["overrides"]["strategy"]
results_a.append(run_experiment(f"A3_{best_strategy}_pesos_clase", **A_COMMON, strategy=best_strategy, class_weights=True))
display(results_table(results_a))

In [ ]:
fig = plot_histories({r["name"]: load_history(r["name"]) for r in results_a}, title="Modelo A: variantes")
plt.show()

**Lectura** (corrida en Colab, T4, semilla 42).

| Variante | Mejor acc en `valid` | Mejor época | Épocas corridas | Minutos |
|---|---|---|---|---|
| A1 feature extraction | 0,9286 | 11 | 16 | 6,1 |
| A2 finetune | 0,9386 | 9 | 14 | 6,2 |
| A3 finetune + pesos de clase | **0,9429** | 7 | 12 | 5,3 |

- **El preentrenamiento hace casi todo el trabajo.** Con el backbone congelado y solo 35.910 parámetros
  entrenables (la cabeza lineal), A1 ya pasa el 0,88 en la primera época y llega a 0,93: las features
  de ImageNet separan bien las razas. Descongelar todo (A2, 11,2 M parámetros) suma un punto.
- **Las diferencias están dentro del ruido.** Con 700 imágenes de `valid` el error estándar de una
  accuracy cercana a 0,93 es de un punto, así que 1 a 1,5 puntos entre variantes no alcanzan para
  decir que una es mejor. Elegimos A3 porque es la mejor en `valid` y no cuesta más, no porque
  sepamos que los pesos de clase ayudan. Tiene sentido que su efecto sea chico: el desbalance de
  `train` es moderado (56 a 185 imágenes por raza) y `valid` está balanceado.
- **Sobreajuste.** En el finetune la accuracy de `train` llega a 0,98 o 0,99 mientras `valid` se
  queda en 0,94: memoriza `train` y por eso el early stopping corta pronto (épocas 12 a 14).
  En A1 la brecha es mucho menor porque la cabeza lineal tiene poca capacidad. La accuracy de
  `train` se mide con augmentation y label smoothing, así que la brecha es solo indicativa.

### Modelo A final

Se reentrena la variante ganadora con `train_classifier`, que guarda `models/resnet18_finetuned.pth` y
su historial (`resnet18_finetuned_history.json`). Recién ahí se evalúa sobre `test`.

In [ ]:
from lib.bootstrap import build_classifier

classifier = build_classifier(settings)


def train_final(results: list[dict], **extra) -> dict:
    """Entrena el modelo final con la mejor variante (una sola vez: queda marcado en Drive)."""
    best = max(results, key=lambda r: r["best_val_accuracy"])
    config = base_config(**(best["overrides"] | extra))
    done = settings.model_path / f"{config.model}.done.json"
    if not done.exists():
        result = classifier.train_classifier(config)
        done.write_text(json.dumps({"from": best["name"], "best_val_accuracy": result.best_val_accuracy,
                                    "best_epoch": result.best_epoch, "config": asdict(config)}, indent=2, default=list))
    classifier.set_active_model(config.model)
    return json.loads(done.read_text())


final_a = train_final(results_a)
final_a

In [ ]:
from lib.evaluation.classification import confusion_matrix, per_class_metrics, top_confused_pairs


def evaluate_final(name: str) -> tuple[dict, dict]:
    classifier.set_active_model(name)
    classifier.load_model(name).to(DEVICE)
    metrics = classifier.evaluate_classifier()  # test, metricas macro
    pred = classifier.predict_split("test")
    np.savez(EXP_DIR / f"test_predictions_{name}.npz", y_true=pred["y_true"], y_pred=pred["y_pred"], probs=pred["probs"])
    return metrics, pred


metrics_a, pred_a = evaluate_final("resnet18_finetuned")
pd.Series(metrics_a).round(4)

### Curvas de entrenamiento y matriz de confusión (Modelo A)

In [ ]:
plot_histories({"resnet18_finetuned": json.loads(classifier.history_path("resnet18_finetuned").read_text())},
               title="Modelo A final")
plt.show()


def confusion_report(pred: dict, title: str) -> None:
    cm = confusion_matrix(pred["y_true"], pred["y_pred"], len(pred["classes"]))
    plot_confusion(cm, pred["classes"], title=title)
    plt.show()
    display(pd.DataFrame(top_confused_pairs(cm, pred["classes"], k=10)))
    recall = pd.Series(per_class_metrics(cm)["recall"], index=pred["classes"]).sort_values()
    print("Razas con menor recall en test (n=10 por raza, descriptivo):", recall.head(8).round(2).to_dict())


confusion_report(pred_a, "Modelo A: matriz de confusion normalizada (test)")

**Lectura.** El modelo final (A3 reentrenado, mejor época 7) da en `test` accuracy 0,9243, precision
0,9333, recall 0,9243, especificidad 0,9989 y F1 0,9230. Queda unos dos puntos por debajo de su
mejor `valid` (0,9429). Es lo esperable: `valid` se usó para elegir la variante y la época, así que
su número es optimista. Por eso el resultado que se informa es el de `test`, medido una sola vez.

Los pares más confundidos en `test` son Boston Terrier y Bulldog (6 errores, casi todos de Bulldog
predicho como Boston Terrier), Cockapoo y Labradoodle (4), y Lhasa con Shih-Tzu y Pastor Alemán con
Malinois (3 cada uno). **Son los mismos pares que el EDA encontró en el espacio del baseline.**
Border Collie y Collie no aparecen entre los diez primeros. Si el baseline sin entrenar y el modelo
ajustado fallan en los mismos pares, el error viene sobre todo del parecido real entre las razas
(cruzas como Cockapoo y Labradoodle, o razas que se distinguen por el pelaje o el tamaño) y de
etiquetas dudosas, más que del modelo. La raza con menor recall es Bulldog (0,3) y le sigue Cockapoo
(0,6). Con 10 imágenes por raza en `test`, el recall por raza es descriptivo: un error mueve 10 puntos.

## 7. Modelo B (obligatorio): CNN propia

**Arquitectura** (`src/lib/models/cnn_custom.py`, plan D-22): 5 bloques conv 3x3, BatchNorm, ReLU y
max-pool 2x2 con 32, 64, 128, 256 y 256 canales; *global average pooling*; capa lineal de embedding
de 256 con ReLU; dropout 0,3; clasificador lineal de 70. Se entrena desde cero (inicialización de Kaiming).

- **Por qué así**: con ~100 imágenes por raza y sin preentrenamiento el riesgo principal es el
  sobreajuste. BatchNorm estabiliza y acelera el entrenamiento; GAP en vez de aplanar reduce mucho los
  parámetros (aplanar 256x7x7 a 256 serían 3,2 M solo en esa capa) y da algo de invariancia a la
  posición; el dropout regulariza la cabeza; la capa de embedding expone la representación para la
  búsqueda por similitud.
- LR 1e-3 con AdamW; hasta 60 épocas en el modelo final (desde cero tarda más en converger que el A).

In [ ]:
from lib.models.cnn_custom import CustomCNN

cnn = CustomCNN(CLASSES)
shapes, x = [], torch.zeros(1, 3, settings.image_size, settings.image_size)
for name, block in [*(("bloque %d" % (i + 1), b) for i, b in enumerate(cnn.features)), ("gap", cnn.pool)]:
    x = block(x)
    shapes.append({"capa": name, "salida": tuple(x.shape[1:]), "parametros": sum(p.numel() for p in block.parameters())})
shapes.append({"capa": "embedding", "salida": (cnn.embedding_dim,), "parametros": sum(p.numel() for p in cnn.embedding.parameters())})
shapes.append({"capa": "clasificador", "salida": (cnn.num_classes,), "parametros": sum(p.numel() for p in cnn.classifier.parameters())})
arch = pd.DataFrame(shapes).set_index("capa")
display(arch)
print(f"Total: {sum(p.numel() for p in cnn.parameters()):,} parametros")

**Tabla de experimentos** (hasta 30 épocas cada uno para entrar en el presupuesto; el final va a 60).
Cada variante cambia **una sola cosa** respecto de la base, para poder atribuirle el efecto:

| Variante | Cambio | Hipótesis |
|---|---|---|
| B1 base | — | referencia |
| B2 dropout 0,5 | más regularización en la cabeza | si B1 sobreajusta (train >> valid), debería ayudar |
| B3 augmentation fuerte | recortes de hasta 25 %, rotación de 20°, más brillo y contraste, más probables | más variedad simula más datos; si es excesiva, la red no llega a aprender (subajuste) |
| B4 estadísticas de `train` | la red re-normaliza la entrada con la media y el desvío de `train` | con BatchNorm después de la primera conv el efecto debería ser chico: la conv absorbe el cambio afín de la entrada |
| B5 sin BatchNorm | quita las capas BatchNorm | entrenamiento más lento e inestable desde cero; muestra para qué está BatchNorm |

**Por qué normalizar la entrada (contexto de B4).** Restar la media y dividir por el desvío de cada canal
deja los canales en escalas comparables (ninguno domina el gradiente) y el problema de optimización queda
mejor condicionado. Con un modelo preentrenado hay que usar las estadísticas con las que se entrenó: el
Modelo A las lee de los pesos de torchvision (`IMAGENET1K_V1.transforms()`) y verifica que coincidan con las
del preprocesamiento compartido; si no, el modelo no se construye. Entrenando desde cero, lo natural es
calcularlas sobre el propio `train`, que es lo que prueba B4 contra la base (ImageNet).

B4 no rompe el pipeline único: la imagen sigue llegando normalizada con ImageNet y la renormalización
vive dentro del modelo (buffers del checkpoint), así la app no cambia.

In [ ]:
B_EPOCHS = 30
STRONG_AUG = dict(crop_border=0.25, crop_p=0.7, rotate_degrees=20.0, rotate_p=0.6, brightness=0.3, contrast=0.3, color_p=0.7)
B_COMMON = dict(model="cnn_custom", epochs=B_EPOCHS)
results_b = [
    run_experiment("B1_base", **B_COMMON),
    run_experiment("B2_dropout_0.5", **B_COMMON, cnn_dropout=0.5),
    run_experiment("B3_aug_fuerte", **B_COMMON, augmentation=STRONG_AUG),
    run_experiment("B4_stats_train", **B_COMMON, cnn_train_stats=True),
    run_experiment("B5_sin_batchnorm", **B_COMMON, cnn_batchnorm=False),
]
display(results_table(results_b))

In [ ]:
fig = plot_histories({r["name"]: load_history(r["name"]) for r in results_b}, title="Modelo B: variantes")
plt.show()

**Lectura.**

| Variante | Mejor acc en `valid` | Mejor época |
|---|---|---|
| B1 base | **0,5571** | 29 |
| B4 estadísticas de `train` | 0,5529 | 28 |
| B5 sin BatchNorm | 0,5443 | 26 |
| B2 dropout 0,5 | 0,4771 | 28 |
| B3 augmentation fuerte | 0,4571 | 27 |

- **Ninguna variante terminó de aprender en 30 épocas.** Todas tienen la mejor época al final, y
  las curvas de `valid` siguen subiendo. La comparación es entonces "con el mismo presupuesto", no
  "en convergencia", y eso condiciona cómo se lee cada hipótesis.
- **B2 y B3 empeoran porque el modelo subajusta, no porque sobreajuste.** La hipótesis de B2 pedía
  que B1 sobreajustara, y eso no pasa: la accuracy de `train` de B1 queda pegada a la de `valid`.
  Más dropout o más augmentation regularizan a un modelo que todavía no aprendió lo suficiente, y
  lo frenan: en B2 y B3 la accuracy de `train` (medida con dropout y augmentation) queda incluso
  por debajo de la de `valid`. Con más épocas la augmentation fuerte podría pagar, pero con este
  presupuesto no.
- **B4 confirma la teoría de la normalización de entrada.** Usar la media y el desvío de `train` en
  vez de los de ImageNet cambia 0,4 puntos, que es ruido: la BatchNorm después de la primera
  convolución ya re-centra y re-escala las activaciones, así que la red se adapta igual a cualquiera
  de las dos normalizaciones. Por eso el pipeline único con estadísticas de ImageNet no le cuesta
  nada al Modelo B.
- **B5 sin BatchNorm queda a un punto de B1.** Esperábamos que sin BatchNorm el entrenamiento fuera
  más lento e inestable. Con cinco bloques, inicialización de Kaiming y AdamW la red entrena igual, y
  la diferencia está dentro del ruido (error estándar de unos 1,9 puntos con accuracy de 0,55). Nos
  quedamos con BatchNorm porque no rinde peor y hace al modelo menos sensible a la escala de la entrada.
- Como era de esperar, el Modelo B queda muy por debajo del A: no tiene preentrenamiento y aprende
  todo de unas 7.200 imágenes.

La variante final es **B1**, reentrenada con un tope de 60 épocas, ya que la diferencia más clara es
que con 30 no le alcanzaba.

### Modelo B final

La mejor variante, con hasta 60 épocas, entrenada con `train_classifier` (`models/cnn_custom.pth`).

In [ ]:
final_b = train_final(results_b, epochs=60)
metrics_b, pred_b = evaluate_final("cnn_custom")
pd.Series(metrics_b).round(4)

### Curvas de entrenamiento y matriz de confusión (Modelo B)

In [ ]:
plot_histories({"cnn_custom": json.loads(classifier.history_path("cnn_custom").read_text())}, title="Modelo B final")
plt.show()
confusion_report(pred_b, "Modelo B: matriz de confusion normalizada (test)")

**Lectura.**

- **Más épocas era lo que faltaba.** Con un tope de 60, B1 llega a 0,7157 en `valid` en la época 54
  (contra 0,5571 con 30), y el early stopping corta en la 59.
- **El modelo sigue sin sobreajustar.** En la mejor época la accuracy de `train` es 0,735 y la de
  `valid` 0,716: está cerca de su límite de capacidad con estos datos, no memorizando.
- **Resultados en `test`:** accuracy 0,7029, precision 0,7213, recall 0,7029, especificidad 0,9957 y
  F1 0,6983. Igual que en el Modelo A, queda algo por debajo de `valid`.
- **Los errores están más repartidos que en el A.** La matriz de confusión tiene mucho más ruido
  fuera de la diagonal.
  - Siguen apareciendo pares del EDA, como Cockapoo con Labradoodle (5 errores, el primero) y Lhasa
    con Shih-Tzu.
  - Aparecen confusiones nuevas entre razas que no se parecen tanto: Cairn y Yorkie, Basenji y Corgi,
    Bull Mastiff y Pug.
  - La lectura es que el Modelo B se apoya en señales más gruesas, como el color, el pelaje o el
    tamaño aparente.
  - Las razas con menor recall son Cocker y Chihuahua (0,2 cada una).

## 8. Evaluación sobre el conjunto independiente

`test` sale del mismo dataset que `train`: imágenes de 224 px, con el perro centrado y ocupando casi
todo el cuadro. Por eso su accuracy mide la generalización **dentro de ese dominio**. El conjunto
independiente (`data/splits/independent_v1.csv`, armado en `1-pre-processing.ipynb`) son fotos reales
de otras fuentes: el perro ocupa una parte de la imagen, hay fondos, personas, otras resoluciones e
iluminaciones. La caída de accuracy entre `test` y el independiente es el **cambio de dominio**.

Para separar sus causas se clasifica cada imagen de dos formas:

- **Imagen completa**: el cambio de dominio "crudo", con el encuadre incluido.
- **Recorte de YOLO** (el de mayor confianza, como hace la Etapa 3): lo que ve de verdad la app.
  Si YOLO no encuentra el perro cuenta como error (la app no mostraría ninguna raza), y se informa
  aparte cuántas veces pasa.

La diferencia entre las dos mide cuánto de la caída explica el encuadre. Se usan solo las imágenes de
un perro; las escenas con varios perros y sin perros son para la detección (sección 12). Con unas pocas
imágenes por raza el intervalo de confianza es ancho y la tabla por raza es descriptiva.

In [ ]:
from lib.dataset.independent import load_registry
from lib.dataset.torch_dataset import read_bgr
from lib.models.inference import predict_proba_fn
from lib.services.detection_service import DetectionService

independent = load_registry(settings.data_path / "splits" / "independent_v1.csv")
single = independent[independent["scene"] == "single"].reset_index(drop=True)
on_disk = single["path"].map(lambda p: (DATASET / p).exists())
print(f"{len(single)} imagenes de un perro en el registro ({single['breed'].nunique()} razas), {on_disk.sum()} en disco")
single = single[on_disk].reset_index(drop=True)

# Detector con la configuracion de la app (.env); la eleccion de sus valores esta en la seccion 12.
detector = DetectionService(classifier, settings.yolo_model, settings.yolo_conf_threshold,
                            settings.yolo_dog_class_id, min_confidence=0.0)
MODELS = {"Modelo A": "resnet18_finetuned", "Modelo B": "cnn_custom"}
predictors = {label: predict_proba_fn(classifier.load_model(name).to(DEVICE), settings.image_size)
              for label, name in MODELS.items()}

records = []
for row in single.itertuples():
    image = read_bgr(DATASET / row.path)
    detections = detector.detect_dogs(image)
    crop = None
    if detections:
        (x1, y1, x2, y2), _ = detections[0]
        crop = image[y1:y2, x1:x2]
    for label, predict in predictors.items():
        for mode, img in (("imagen completa", image), ("recorte YOLO", crop)):
            probs = predict([img])[0] if img is not None else None
            records.append({"path": row.path, "breed": row.breed, "modelo": label, "modo": mode,
                            "detectado": crop is not None,
                            "pred": CLASSES[int(probs.argmax())] if probs is not None else "sin deteccion",
                            "prob": float(probs.max()) if probs is not None else 0.0})
indep = pd.DataFrame(records)
indep["correct"] = indep["pred"] == indep["breed"]
missed = int((~indep.drop_duplicates("path")["detectado"]).sum())
print(f"YOLO no detecto ningun perro en {missed} de {len(single)} imagenes")

In [ ]:
from lib.evaluation.classification import bootstrap_ci, classification_metrics, mcnemar_test

index = {b: i for i, b in enumerate(CLASSES)} | {"sin deteccion": len(CLASSES)}
rows = []
for (label, mode), group in indep.groupby(["modelo", "modo"], sort=False):
    metrics = classification_metrics(group["breed"].map(index).to_numpy(), group["pred"].map(index).to_numpy(), len(CLASSES) + 1)
    _, low, high = bootstrap_ci(group["correct"].to_numpy())
    rows.append({"modelo": label, "modo": mode, "imagenes": len(group), **{k: round(v, 4) for k, v in metrics.items()},
                 "accuracy IC95": f"[{low:.3f}, {high:.3f}]"})
for label, metrics in (("Modelo A", metrics_a), ("Modelo B", metrics_b)):
    rows.append({"modelo": label, "modo": "test (referencia)", "imagenes": 700, **{k: round(v, 4) for k, v in metrics.items()}})
display(pd.DataFrame(rows).set_index(["modelo", "modo"]))

for label in MODELS:
    pivot = indep[indep["modelo"] == label].pivot(index="path", columns="modo", values="correct")
    test = mcnemar_test(pivot["recorte YOLO"].to_numpy(), pivot["imagen completa"].to_numpy())
    print(f"{label}, recorte contra imagen completa: solo el recorte acierta {test['only_a']}, "
          f"solo la completa acierta {test['only_b']}, p = {test['p_value']:.3f}")

In [ ]:
by_breed = (indep[indep["modo"] == "recorte YOLO"].pivot_table(index="breed", columns="modelo", values="correct", aggfunc="mean")
            .assign(n=single.groupby("breed").size()).sort_values("Modelo A"))
print("Accuracy por raza con el recorte de YOLO (descriptivo: pocas imagenes por raza)")
display(by_breed.round(2))

from lib.visualization.grid import show_grid

sample = indep[(indep["modelo"] == "Modelo A") & (indep["modo"] == "recorte YOLO")].sample(min(16, len(single)), random_state=SEED)
fig = show_grid([DATASET / p for p in sample["path"]],
                [f"{b[:12]}\n-> {p[:12]} {q:.2f}" for b, p, q in zip(sample["breed"], sample["pred"], sample["prob"])],
                cols=8, size=160, cell_inches=1.6, suptitle="Conjunto independiente: raza real -> prediccion del Modelo A (recorte)")
plt.show()

**Lectura.** TO-DO al tener el conjunto independiente completo (Lucas está sumando 7 razas).

## 9. Embeddings de los modelos entrenados

Los modelos de la Etapa 2 también sirven para la búsqueda por similitud de la Etapa 1: su penúltima
capa es un embedding (512 dimensiones en el Modelo A, 256 en el B), que la app usa cuando
`EMBEDDING_MODEL` no es `baseline` (`extract_custom_embedding`). La pregunta es si entrenar para
clasificar razas mejora el espacio de embeddings para buscar razas.

**Qué se espera.** La pérdida de clasificación empuja a que las imágenes de una raza queden del mismo lado
de la frontera lineal de la cabeza, no necesariamente juntas: la entropía cruzada separa por *dirección*
y no penaliza que una raza esté dispersa. Aun así, en la práctica las clases quedan agrupadas, y como
los vectores se normalizan en L2 la similitud coseno compara justamente direcciones. El baseline
(ResNet50 de ImageNet, sin ajustar) tiene un backbone más grande y ya vio 118 razas de perro en ImageNet.

Mismo protocolo que la sección 5: galería = `train`, consultas = `test`, coseno, NDCG@10 con
`n_relevant` = imágenes de la raza en la galería, raza por voto de los 10 vecinos y sin umbral. Se suma
el **coeficiente de silueta** (coseno) de los embeddings de `test`: para cada imagen compara la distancia
media a su raza con la de la raza vecina más cercana; va de -1 a 1 y mide cuán compactas y separadas
quedan las razas, sin depender de la galería.

In [ ]:
from sklearn.metrics import silhouette_score

from lib.evaluation.retrieval import evaluate_queries, load_or_compute_embeddings, per_breed_table, similarity_matrix
from lib.models.baseline import BaselineEmbedder
from lib.models.inference import embed_batch_fn

gallery = manifest[manifest["split"] == "train"].reset_index(drop=True)
queries = manifest[manifest["split"] == "test"].reset_index(drop=True)
EMB_DIR = EXP_DIR / "embeddings"


def split_embeddings(key: str, embed_batch) -> dict[str, np.ndarray]:
    return {split: load_or_compute_embeddings(EMB_DIR / f"{key}_{split}.npz", rows["path"].tolist(),
                                              [DATASET / p for p in rows["path"]], embed_batch)
            for split, rows in (("train", gallery), ("test", queries))}


EMBEDDERS = {
    "baseline (ResNet50 ImageNet)": ("baseline", lambda: BaselineEmbedder(settings.image_size, DEVICE).embed_batch),
    "Modelo A (ResNet18 fine-tuned)": ("resnet18_finetuned", lambda: embed_batch_fn(classifier.load_model("resnet18_finetuned").to(DEVICE), settings.image_size)),
    "Modelo B (CNN propia)": ("cnn_custom", lambda: embed_batch_fn(classifier.load_model("cnn_custom").to(DEVICE), settings.image_size)),
}
embeddings, retrieval_eval, rows = {}, {}, []
for label, (key, make) in EMBEDDERS.items():
    emb = embeddings[label] = split_embeddings(key, make())
    evaluation = retrieval_eval[label] = evaluate_queries(similarity_matrix(emb["test"], emb["train"], "cosine"),
                                                          queries["breed"], gallery["breed"])
    summary = evaluation.summary()
    rows.append({"embeddings": label, "dimension": emb["train"].shape[1],
                 **{k: round(v, 4) for k, v in summary.items() if k not in ("n", "rejected")},
                 "silueta (test)": round(float(silhouette_score(emb["test"], queries["breed"], metric="cosine")), 4)})
emb_table = pd.DataFrame(rows).set_index("embeddings")
display(emb_table)

In [ ]:
# Comparacion pareada con el baseline: las mismas 700 consultas, asi que se compara consulta a consulta.
from lib.evaluation.classification import mcnemar_test

base = retrieval_eval["baseline (ResNet50 ImageNet)"].per_query
rng = np.random.default_rng(SEED)
rows = []
for label in list(EMBEDDERS)[1:]:
    other = retrieval_eval[label].per_query
    diff = (other["ndcg"] - base["ndcg"]).to_numpy()
    boot = diff[rng.integers(0, len(diff), size=(2000, len(diff)))].mean(axis=1)
    test = mcnemar_test(other["correct"].to_numpy(), base["correct"].to_numpy())
    rows.append({"embeddings": label, "NDCG@10 menos baseline": round(diff.mean(), 4),
                 "IC95": f"[{np.quantile(boot, 0.025):.4f}, {np.quantile(boot, 0.975):.4f}]",
                 "accuracy: solo este acierta": test["only_a"], "solo el baseline acierta": test["only_b"],
                 "McNemar p": round(test["p_value"], 4)})
display(pd.DataFrame(rows).set_index("embeddings"))

In [ ]:
worst = pd.concat({label: per_breed_table(ev.per_query)["ndcg"] for label, ev in retrieval_eval.items()}, axis=1)
print("NDCG@10 por raza: las 10 peores con el Modelo A (descriptivo, 10 consultas por raza)")
display(worst.sort_values("Modelo A (ResNet18 fine-tuned)").head(10).round(3))

**Visualización.** t-SNE de los embeddings de `test` de 12 razas: los pares que más se confunden y
algunas razas bien distintas. Siguiendo class_4 (6.14.3), primero PCA a 50 dimensiones (quita ruido
y acelera) y después t-SNE a 2. t-SNE conserva vecindarios, no distancias globales: la separación entre
grupos lejanos y el tamaño de cada grupo no se interpretan, sí qué razas quedan mezcladas.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SHOWN = ["Boston Terrier", "Bulldog", "Cockapoo", "Labradoodle", "Lhasa", "Shih-Tzu",
         "German Sheperd", "Malinois", "Border Collie", "Collie", "Dalmation", "Pug"]
mask = queries["breed"].isin(SHOWN).to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, emb) in zip(axes, embeddings.items()):
    x = emb["test"][mask]
    x = PCA(n_components=min(50, x.shape[1]), random_state=SEED).fit_transform(x)
    xy = TSNE(n_components=2, perplexity=20, init="pca", max_iter=1000, random_state=SEED).fit_transform(x)
    for k, breed in enumerate(SHOWN):
        sel = queries["breed"].to_numpy()[mask] == breed
        ax.scatter(xy[sel, 0], xy[sel, 1], s=18, color=plt.cm.tab20(k), label=breed)
    ax.set(title=label, xticks=[], yticks=[])
axes[-1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.suptitle("t-SNE de los embeddings de test (12 razas)")
fig.tight_layout()
plt.show()

**Lectura** (corrida local en CPU; los embeddings no dependen del dispositivo).

| Embeddings | Dimensión | NDCG@10 | precision@10 | accuracy por vecinos | Silueta (test) |
|---|---|---|---|---|---|
| baseline (ResNet50 ImageNet) | 2048 | 0,902 [0,887; 0,917] | 0,896 | 0,943 [0,924; 0,960] | 0,297 |
| Modelo A (ResNet18 fine-tuned) | 512 | **0,919** [0,902; 0,934] | 0,918 | 0,940 [0,923; 0,957] | 0,303 |
| Modelo B (CNN propia) | 256 | 0,596 [0,570; 0,623] | 0,585 | 0,704 [0,670; 0,739] | 0,060 |

- **El Modelo A ordena mejor a los vecinos que el baseline, con un vector cuatro veces más chico.**
  Comparando consulta a consulta, su NDCG@10 es 0,017 más alto (IC 95 % [0,004; 0,030]): la mejora es
  chica pero no es ruido. En cambio, la raza votada por los 10 vecinos acierta lo mismo (McNemar: 19
  consultas que solo acierta A y 21 que solo acierta el baseline, p = 0,87). Es decir: el fine-tuning
  sube más vecinos correctos al principio de la lista, pero el voto ya acertaba igual con el baseline.
- **El fine-tuning reordena el espacio donde más se necesitaba.** Mejoran las razas que el baseline
  confundía: Bulldog (NDCG 0,42 a 0,58), Cockapoo (0,57 a 0,75), Lhasa (0,61 a 0,78) y Boston Terrier
  (0,62 a 0,80). A cambio empeoran algunas que el baseline resolvía bien, como Malinois (0,80 a 0,63) y
  Schnauzer (0,91 a 0,71). Con 10 consultas por raza esto es descriptivo.
- **El Modelo B casi no agrupa las razas.** La silueta de 0,06 indica razas muy mezcladas (la del
  baseline y la del A, alrededor de 0,30, es baja en valor absoluto, como es habitual con 70 clases en
  cientos de dimensiones, pero cinco veces mayor). Su accuracy por vecinos (0,704) es la misma que la de
  su clasificador (0,703): el embedding sabe lo mismo que la cabeza, que es poco.
- **t-SNE.** En los tres modelos los pares parecidos quedan uno al lado del otro: Cockapoo y Labradoodle,
  Lhasa y Shih-Tzu, Pastor Alemán y Malinois, Boston Terrier y Bulldog. Lo que cambia es la mezcla: en el
  baseline y en el A cada par forma un grupo compacto y casi sin intrusos de otras razas, y en el B los
  grupos se estiran y se tocan con razas que no se parecen.

## 10. Análisis de errores

Sobre `test` y con el Modelo A (el que usa la app). En multiclase, un error es a la vez un **falso
negativo** de la raza real (no la reconoció) y un **falso positivo** de la raza predicha (la nombró sin
que estuviera). Se miran tres cosas:

1. **Errores con alta confianza**: los más peligrosos, porque ningún umbral de confianza los filtra.
2. **Las razas con peor F1** en los dos modelos.
3. **Grad-CAM** (propio, `lib/evaluation/gradcam.py`, D-10) para ver en qué zona de la imagen se apoya cada
   modelo: si mira al perro o al fondo. Grad-CAM pondera cada mapa de la última capa convolucional por el
   promedio de los gradientes del logit de la clase, suma y aplica ReLU; el resultado se reescala a la
   imagen. Se muestra sobre lo que ve el modelo (la imagen con el padding del preprocesamiento).

In [ ]:
from lib.dataset.torch_dataset import read_bgr
from lib.visualization.grid import show_grid

MODELS = {"Modelo A": "resnet18_finetuned", "Modelo B": "cnn_custom"}
probs_a = pred_a["probs"]
errors = pd.DataFrame({
    "path": pred_a["rows"]["path"].to_numpy(),
    "real": np.array(CLASSES)[pred_a["y_true"]], "predicha": np.array(CLASSES)[pred_a["y_pred"]],
    "prob predicha": probs_a.max(axis=1), "prob real": probs_a[np.arange(len(probs_a)), pred_a["y_true"]],
})
errors = errors[errors["real"] != errors["predicha"]].sort_values("prob predicha", ascending=False)
print(f"{len(errors)} errores del Modelo A en test; con probabilidad >= 0.5: {(errors['prob predicha'] >= 0.5).sum()}")
display(errors.head(12).round(3))
fig = show_grid([DATASET / p for p in errors["path"].head(16)],
                [f"{r[:12]} -> {p[:12]}\n{q:.2f}" for r, p, q in zip(errors["real"], errors["predicha"], errors["prob predicha"])][:16],
                cols=8, size=160, cell_inches=1.6, suptitle="Errores del Modelo A con mayor confianza (real -> predicha, probabilidad)")
plt.show()

In [ ]:
f1 = pd.DataFrame({label: per_class_metrics(confusion_matrix(p["y_true"], p["y_pred"], len(CLASSES)))["f1"]
                   for label, p in (("Modelo A", pred_a), ("Modelo B", pred_b))}, index=CLASSES)
print("Razas con menor F1 en test (10 imagenes por raza: descriptivo)")
display(f1.sort_values("Modelo A").head(8).round(2))
display(f1.sort_values("Modelo B").head(8).round(2))

In [ ]:
from lib.evaluation.gradcam import GradCAM
from lib.visualization.explain import model_input_rgb, overlay_heatmap
from lib.vision.preprocessing import preprocess_to_tensor

hits = np.flatnonzero(pred_a["y_true"] == pred_a["y_pred"])
examples = list(errors["path"].head(4)) + list(pred_a["rows"]["path"].to_numpy()[np.random.default_rng(SEED).choice(hits, 2, replace=False)])

panels, titles = [], []
for path in examples:
    image = read_bgr(DATASET / path)
    base = model_input_rgb(image, settings.image_size)
    x = preprocess_to_tensor(image, settings.image_size).unsqueeze(0)
    true_breed = queries.loc[queries["path"] == path, "breed"].iloc[0]
    panels.append(base); titles.append(f"real: {true_breed[:14]}")
    for label, name in MODELS.items():
        with GradCAM(classifier.load_model(name)) as cam:
            heatmap, index, prob = cam(x)
        panels.append(overlay_heatmap(base, heatmap)); titles.append(f"{label}: {CLASSES[index][:12]} {prob:.2f}")
fig = show_grid(panels, titles, cols=6, size=224, cell_inches=2.0,
                suptitle="Grad-CAM de la clase predicha (4 errores del Modelo A y 2 aciertos)")
plt.show()

**Lectura.**

- **Los errores del Modelo A son casi todos entre razas parecidas.** De los 53 errores en `test`, los de
  mayor confianza son Cockapoo con Labradoodle, Bulldog con Bulldog Francés y Boston Terrier, Shih-Tzu con
  Lhasa, Malinois con Pastor Alemán y los dos perros sin pelo (American y Mexican Hairless): los mismos
  pares del EDA y de la sección 6.
- **18 de los 53 errores tienen probabilidad de 0,5 o más.** Ningún umbral razonable de confianza los
  filtra (sección 12): el modelo está seguro y se equivoca.
- **Parte de los errores son de etiqueta, no del modelo.** A simple vista, la imagen etiquetada Bulldog que
  el modelo llama Boston Terrier con 0,83 tiene la cara y las orejas de un Boston Terrier, y la que llama
  Bulldog Francés con 0,70 tiene las orejas de murciélago del Francés. Es una hipótesis (no somos
  expertos en razas), pero explicaría por qué Bulldog es la raza con peor F1 (0,40): un modelo no puede
  acertar una etiqueta equivocada.
- **Grad-CAM: los dos modelos miran al perro, no al fondo.** En los seis ejemplos el mapa del Modelo A se
  concentra en la cabeza (ojos, hocico, orejas), que es donde están los rasgos de raza; el del Modelo B es
  más difuso y en el Schnauzer se reparte por el pelaje y el borde de la imagen, con una predicción de
  0,21. En estos casos el error no viene de un atajo con el fondo sino del parecido real entre razas. Son
  seis ejemplos: es una ilustración, no una prueba.
- **Las peores razas del Modelo B son otras:** Cocker (F1 0,25), Chihuahua (0,31), Greyhound (0,40) y
  Basenji (0,44), que el Modelo A resuelve con 0,86 o más. Son razas con mucha variación interna (colores,
  pelo largo o corto, poses), y una red sin preentrenamiento que vio unas 100 fotos por raza se apoya en
  señales gruesas como el color o la silueta.

## 11. Costo computacional

Lo que cuesta cada modelo en la app: parámetros, tamaño del archivo, dimensión del embedding y latencia
por imagen **en CPU** (donde corre la app en Docker), con una imagen por llamada como en un request.
Se toman 5 llamadas de calentamiento y la mediana de 30.

Se compara además el `.pth` (lo que se entrega, D-09) con el mismo modelo exportado a **ONNX** y corrido
con onnxruntime, que aplica optimizaciones de grafo (fusiona convolución, BatchNorm y ReLU en una sola
operación) y suele ser más rápido en CPU. El baseline de la Etapa 1 se mide igual (sin cabeza: solo
embedding). La latencia depende de la máquina: el procesador queda impreso abajo.

In [ ]:
from lib.evaluation.cost import count_parameters, cpu_description, file_size_mb, onnx_cpu_latency, torch_cpu_latency
from lib.models.export import export_onnx

print("CPU:", cpu_description())
rows = []
baseline_model = BaselineEmbedder(settings.image_size, "cpu")
baseline_net = baseline_model.model
weights_file = next((Path(os.environ["TORCH_HOME"]) / "hub" / "checkpoints").glob("resnet50-*.pth"), None)
rows.append({"modelo": "baseline (ResNet50)", "parametros": count_parameters(baseline_net),
             "MB .pth": round(file_size_mb(weights_file), 1) if weights_file else None, "dimension": 2048,
             "ms CPU .pth": round(torch_cpu_latency(baseline_net, settings.image_size)["ms_median"], 1)})
for label, name in MODELS.items():
    model = classifier.load_model(name)
    onnx_path = export_onnx(model, EXP_DIR / f"{name}.onnx", settings.image_size)
    rows.append({"modelo": label, "parametros": count_parameters(model),
                 "MB .pth": round(file_size_mb(classifier.checkpoints[name]), 1),
                 "MB .onnx": round(file_size_mb(onnx_path), 1), "dimension": model.embedding_dim,
                 "ms CPU .pth": round(torch_cpu_latency(model, settings.image_size)["ms_median"], 1),
                 "ms CPU .onnx": round(onnx_cpu_latency(onnx_path, settings.image_size)["ms_median"], 1)})
cost_table = pd.DataFrame(rows).set_index("modelo")
display(cost_table)

**Lectura** (medido localmente en un Intel de 4 hilos; en Colab los milisegundos cambian, las proporciones
se mantienen).

| Modelo | Parámetros | MB `.pth` | Dimensión | ms CPU `.pth` | ms CPU `.onnx` |
|---|---|---|---|---|---|
| baseline (ResNet50) | 23,5 M | 97,8 | 2048 | 88,7 | — |
| Modelo A (ResNet18) | 11,2 M | 42,9 | 512 | 39,8 | 23,7 |
| Modelo B (CNN propia) | 1,06 M | 4,1 | 256 | 26,6 | 12,1 |

- **Parámetros no es lo mismo que cómputo.** El Modelo B tiene 10 veces menos parámetros que el A pero
  es solo 1,5 veces más rápido. El tiempo lo dominan las operaciones, no los pesos: el B aplica sus
  primeras convoluciones a la resolución completa (224 px) antes de achicar, unas 0,85 GMAC por imagen,
  mientras que la ResNet18 reduce la imagen 4 veces en la primera capa y hace unas 1,8 GMAC. Los
  parámetros sí deciden el tamaño en disco (4 MB contra 43 MB).
- **ONNX es 1,7 veces más rápido en el A y 2,2 en el B**, con el mismo tamaño: onnxruntime fusiona
  convolución, BatchNorm y ReLU y elige kernels para CPU. Se entrega igual el `.pth` (D-09): la app
  también carga `.onnx`, así que es una mejora disponible si la latencia importara.
- **Frente al baseline de la Etapa 1, el Modelo A da embeddings 2,2 veces más rápido** y cuatro veces más
  chicos (512 contra 2048 floats: 2 KB contra 8 KB por imagen en pgvector).

## 12. Etapa 3: detección y umbral de confianza del clasificador

### Validación de la detección (YOLO)

La Etapa 3 detecta los perros con YOLO preentrenado en COCO (clase 16, sin reentrenar) y clasifica cada
recorte. Se evalúan tres decisiones:

- **`YOLO_CONF_THRESHOLD`** (el enunciado pide mostrar 2 o 3 valores): con uno bajo aparecen detecciones
  falsas (gatos, sombras, un mismo perro dos veces); con uno alto se pierden perros chicos, tapados o de
  espaldas.
- **`imgsz`**: YOLO redimensiona la imagen para que su lado mayor mida `imgsz` (640 por defecto). Una
  imagen de 224 px se agranda casi tres veces y la red, que aprendió con objetos de tamaños de COCO, ve un
  perro gigante y borroso; en una foto grande con perros lejanos pasa lo contrario: al achicarla a 640 los
  perros quedan de pocos píxeles. Por eso se prueban 320, 640 y 1280.
- **YOLOv8n contra YOLO11n** (D-14): la versión 11 promete más precisión con menos parámetros.

**Cómo se mide.** No hay cajas anotadas, pero sí la cantidad de perros de cada imagen: 1 en `test` y en
las fotos de una raza del conjunto independiente, `n_dogs` en las escenas con varios perros y 0 en las
escenas sin perros. Se evalúa el conteo, que es lo que la app necesita (un recorte por perro):
porcentaje de imágenes con algún perro detectado, de conteos exactos, error medio del conteo y
**falsas alarmas** (imágenes sin perros donde detecta alguno). YOLO se corre una vez con un umbral bajo
(0,05) y después se filtra por cada umbral, que es equivalente porque el NMS solo suprime una caja con
otra de mayor confianza. La latencia se mide en CPU.

In [ ]:
from ultralytics import YOLO

from lib.evaluation.detection import count_table, detect_confidences

scenes = independent[independent["path"].map(lambda p: (DATASET / p).exists())]
groups = {
    "test (224 px)": (queries["path"].tolist(), [1] * len(queries)),
    "independiente, un perro": (scenes.loc[scenes["scene"] == "single", "path"].tolist(), [1] * int((scenes["scene"] == "single").sum())),
    "independiente, varios": (scenes.loc[scenes["scene"] == "multiple", "path"].tolist(), scenes.loc[scenes["scene"] == "multiple", "n_dogs"].astype(int).tolist()),
    "independiente, sin perros": (scenes.loc[scenes["scene"] == "no_dog", "path"].tolist(), [0] * int((scenes["scene"] == "no_dog").sum())),
}
print({g: len(p) for g, (p, _) in groups.items()})
images = {g: [read_bgr(DATASET / p) for p in paths] for g, (paths, _) in groups.items()}

THRESHOLDS = [0.15, 0.25, 0.40]
YOLO_DIR = Path(settings.yolo_model).parent
det_tables, latency = {}, []
for weights in ("yolov8n.pt", "yolo11n.pt"):
    yolo = YOLO(str(YOLO_DIR / weights))
    for imgsz in (320, 640, 1280):
        confs, counts, names = [], [], []
        for g, (paths, n) in groups.items():
            c, ms = detect_confidences(yolo, images[g], imgsz=imgsz, iou=settings.yolo_iou_threshold, device="cpu")
            confs += c; counts += n; names += [g] * len(c)
            latency.append({"modelo": weights, "imgsz": imgsz, "grupo": g, "ms CPU (mediana)": round(ms, 1)})
        det_tables[(weights, imgsz)] = count_table(confs, counts, names, THRESHOLDS)

In [ ]:
summary = pd.concat(det_tables, names=["modelo", "imgsz"]).reset_index()
cols = ["detection_rate", "exact_count", "count_mae", "false_alarm"]
for metric in cols:
    print(metric)
    display(summary.pivot_table(index=["modelo", "imgsz", "umbral"], columns="grupo", values=metric).round(3))
display(pd.DataFrame(latency).pivot_table(index=["modelo", "imgsz"], columns="grupo", values="ms CPU (mediana)"))

**Efecto del IoU del NMS** en las escenas con perros superpuestos. El NMS descarta una caja si se solapa
más que `YOLO_IOU_THRESHOLD` con otra de mayor confianza: con un IoU bajo, dos perros pegados pueden
quedar como uno; con uno alto, un mismo perro puede aparecer dos veces.

In [ ]:
CHOSEN = {"weights": "yolov8n.pt", "imgsz": 640, "conf": 0.25}  # TO-DO: fijar con las tablas de arriba
yolo = YOLO(str(YOLO_DIR / CHOSEN["weights"]))
multi_paths, multi_n = groups["independiente, varios"]
rows = []
for iou in (0.45, 0.7):
    c, _ = detect_confidences(yolo, images["independiente, varios"], imgsz=CHOSEN["imgsz"], iou=iou, device="cpu")
    found = np.array([(x >= CHOSEN["conf"]).sum() for x in c])
    rows.append({"iou": iou, "conteo exacto": (found == np.array(multi_n)).mean(),
                 "de menos": (found < np.array(multi_n)).mean(), "de mas": (found > np.array(multi_n)).mean()})
display(pd.DataFrame(rows).set_index("iou").round(3))

In [ ]:
from lib.visualization.explain import draw_boxes

shown = []
for g in ("independiente, varios", "independiente, sin perros", "independiente, un perro"):
    for image in images[g][:3]:
        result = yolo.predict(image, conf=CHOSEN["conf"], iou=settings.yolo_iou_threshold, classes=[settings.yolo_dog_class_id],
                              imgsz=CHOSEN["imgsz"], device="cpu", verbose=False)[0]
        boxes = [(tuple(int(v) for v in b), float(s)) for b, s in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy())]
        shown.append(draw_boxes(image, boxes, max_side=480))
fig = show_grid(shown, cols=3, size=480, cell_inches=4.0, suptitle=f"Detecciones con {CHOSEN}")
plt.show()

**Lectura.** TO-DO al tener las escenas en disco y elegir los valores.

### Umbral de confianza del clasificador (`CLASSIFIER_MIN_CONFIDENCE`, D-27)

En la Etapa 3, si la probabilidad de la raza predicha queda bajo `CLASSIFIER_MIN_CONFIDENCE` la app
responde `unknown` en vez de mostrar una raza dudosa. Es **clasificación selectiva**: se cambia cobertura
(proporción de recortes con respuesta) por precisión de lo que se responde. La probabilidad softmax de
una red profunda suele estar sobreconfiada (Guo et al., 2017), así que no se lee como probabilidad real:
el umbral se **calibra con datos**.

Criterio (D-27): sobre `valid`, la curva de riesgo-cobertura (accuracy de lo aceptado contra proporción
aceptada) y el **menor umbral con accuracy aceptada >= 95 %**; el menor porque es el que más cobertura
conserva. Después se mide una sola vez en `test` con ese umbral fijo. `valid` ya se usó para elegir la
variante y la época, así que su accuracy es algo optimista; por eso se confirma en `test`.

In [ ]:
from lib.evaluation.classification import apply_confidence_threshold, confidence_threshold, risk_coverage

fig, ax = plt.subplots(figsize=(6.5, 4))
rows = []
for (label, name), test_pred in zip(MODELS.items(), (pred_a, pred_b)):
    classifier.set_active_model(name)
    valid_pred = classifier.predict_split("valid")
    curve = risk_coverage(valid_pred["probs"], valid_pred["y_true"])
    chosen = confidence_threshold(valid_pred["probs"], valid_pred["y_true"], target_accuracy=0.95)
    applied = apply_confidence_threshold(test_pred["probs"], test_pred["y_true"], chosen["threshold"])
    ax.plot(curve["coverage"], curve["accuracy"], label=label)
    ax.scatter([chosen["coverage"]], [chosen["accuracy"]], zorder=3)
    rows.append({"modelo": label, "umbral (valid)": round(chosen["threshold"], 3),
                 "cobertura valid": round(chosen["coverage"], 3), "accuracy aceptada valid": round(chosen["accuracy"], 3),
                 "cobertura test": round(applied["coverage"], 3), "accuracy aceptada test": round(applied["accuracy"], 3),
                 "errores rechazados test": round(applied["errors_rejected"], 3)})
classifier.set_active_model("resnet18_finetuned")
ax.axhline(0.95, color="gray", linestyle=":", linewidth=1)
ax.set(xlabel="cobertura (proporcion aceptada)", ylabel="accuracy de lo aceptado", title="Riesgo-cobertura en valid")
ax.legend()
plt.show()
min_conf_table = pd.DataFrame(rows).set_index("modelo")
display(min_conf_table)

**Lectura.**

| Modelo | Umbral (valid) | Cobertura valid | Accuracy aceptada valid | Cobertura test | Accuracy aceptada test | Errores rechazados test |
|---|---|---|---|---|---|---|
| A | 0,294 | 0,973 | 0,950 | 0,961 | 0,948 | 34 % |
| B | 0,654 | 0,346 | 0,950 | 0,314 | 0,936 | 93 % |

- **Para el Modelo A alcanza un umbral bajo.** Sin umbral ya acierta el 94,3 % de `valid`; con 0,294
  deja sin respuesta el 2,7 % de `valid` (3,9 % de `test`) y con eso descarta un tercio de los errores.
  En `test` la accuracy aceptada queda en 0,948, apenas bajo el 0,95 buscado: era esperable, porque
  `valid` ya se había usado para elegir la variante y la época y su número es algo optimista.
- **Para el Modelo B el criterio lo vuelve inútil:** para llegar al 95 % tiene que rechazar dos de cada
  tres imágenes. Otra razón para que la Etapa 3 use el Modelo A.
- **El umbral no arregla los errores seguros:** los 18 errores con probabilidad de 0,5 o más (sección
  10) pasan cualquier umbral razonable. Subirlo a 0,5 rechazaría el 66 % de los errores pero también el
  14 % de las imágenes.
- **Valor propuesto: `CLASSIFIER_MIN_CONFIDENCE=0.30`** con el Modelo A (el umbral de `valid`
  redondeado). Queda una salvedad: se calibró con imágenes del dataset, y en la Etapa 3 el clasificador
  recibe recortes de YOLO de fotos reales, donde es de esperar que la confianza baje y aparezcan más
  `unknown`. Se revisa con el conjunto independiente (sección 8).

## 13. Comparación y elección final

### Modelo A contra Modelo B sobre `test`

Las dos predicciones son sobre las **mismas** 700 imágenes, así que se comparan de forma pareada:

- **Intervalo bootstrap del 95 %** de la accuracy de cada modelo (2.000 remuestreos de las 700 imágenes).
- **Prueba de McNemar** (exacta): solo cuentan las imágenes donde un modelo acierta y el otro no. Si
  esos dos conteos son muy distintos, la diferencia no es casualidad. Es más sensible que comparar los
  dos intervalos, porque usa el emparejamiento.

In [ ]:
from lib.evaluation.classification import bootstrap_ci, mcnemar_test

correct_a = pred_a["y_true"] == pred_a["y_pred"]
correct_b = pred_b["y_true"] == pred_b["y_pred"]
rows = []
for name, metrics, correct in [("Modelo A (ResNet18)", metrics_a, correct_a), ("Modelo B (CNN propia)", metrics_b, correct_b)]:
    _, low, high = bootstrap_ci(correct)
    rows.append({"modelo": name, **{k: round(v, 4) for k, v in metrics.items()}, "accuracy IC95": f"[{low:.3f}, {high:.3f}]"})
display(pd.DataFrame(rows).set_index("modelo"))
test = mcnemar_test(correct_a, correct_b)
print(f"McNemar: solo A acierta {test['only_a']}, solo B acierta {test['only_b']}, p = {test['p_value']:.2e}")

**Lectura.**

| Modelo | accuracy en `test` | IC 95 % (bootstrap) | F1 |
|---|---|---|---|
| A (ResNet18 finetune + pesos de clase) | 0,9243 | [0,904; 0,943] | 0,9230 |
| B (CNN propia) | 0,7029 | [0,670; 0,739] | 0,6983 |

- **La diferencia es clara.** Los intervalos no se superponen, y McNemar lo confirma: hay 175
  imágenes que solo acierta A y 20 que solo acierta B (p = 4,3e-32).
- **No es solo que A acierte más: B casi no aporta aciertos propios.** Las 20 imágenes que solo
  acierta B son el 3 % de `test`, así que combinar los dos modelos tendría poco para ganar.
- **Para clasificar, la app usa el Modelo A.**

### Tabla final y elección

Todo junto: la Etapa 1 (búsqueda por vecinos, sección 9), la Etapa 2 (clasificador sobre `test`) y el
costo (sección 11). La accuracy "por vecinos" es la de la raza votada por los 10 vecinos más parecidos de
`train`, sin umbral; la "del clasificador" es la de la cabeza entrenada.

In [ ]:
final = pd.DataFrame({
    "accuracy test (clasificador)": [None, metrics_a["accuracy"], metrics_b["accuracy"]],
    "F1 test (clasificador)": [None, metrics_a["f1"], metrics_b["f1"]],
    "accuracy test (vecinos)": emb_table["accuracy"].to_numpy(),
    "NDCG@10": emb_table["ndcg@10"].to_numpy(),
    "silueta": emb_table["silueta (test)"].to_numpy(),
}, index=["baseline (ResNet50)", "Modelo A", "Modelo B"])
final = final.join(cost_table)
display(final.round(4))

**Elección.**

- **Clasificación (Etapas 2 y 3): Modelo A.** Acierta el 92,4 % de `test` contra el 70,3 % del B, la
  diferencia es significativa (McNemar, sección 13) y con `CLASSIFIER_MIN_CONFIDENCE=0.30` responde el
  96 % de las imágenes con 94,8 % de acierto. El Modelo B es diez veces más liviano, pero 40 ms contra
  27 ms por imagen no justifican perder 22 puntos de accuracy en esta app.
- **Búsqueda (`EMBEDDING_MODEL`): propuesta `resnet18_finetuned`**, a confirmar con el Bloque A porque
  cambia la Etapa 1. Ordena mejor a los vecinos que el baseline (NDCG@10 +0,017, IC 95 % [0,004; 0,030])
  con la misma accuracy por vecinos, el vector es cuatro veces más chico, el embedding es 2,2 veces más
  rápido y la app carga un solo modelo para las tres etapas. El costo: hay que reindexar la galería en la
  tabla de ese modelo (I.1) y calibrar el umbral de `unknown` de la Etapa 1 con sus embeddings, porque la
  escala de los scores cambia de modelo a modelo. Si no llegamos con eso, el baseline es una opción
  válida: su accuracy por vecinos es la misma.

**Búsqueda por similitud contra clasificación supervisada.** Con el Modelo A las dos aciertan parecido:
0,940 por vecinos y 0,924 con la cabeza, con intervalos que se superponen.

- **Búsqueda por similitud.** Para sumar una raza alcanza con indexar sus fotos, sin reentrenar; la
  respuesta se explica sola (estas son las imágenes más parecidas) y el umbral sobre el mejor vecino da un
  rechazo natural. A cambio, guarda y recorre toda la galería (crece con cada imagen), depende de que la
  galería esté bien etiquetada y el umbral se calibra para cada modelo de embeddings.
- **Clasificación supervisada.** Una sola pasada por la red, sin base de datos, y una probabilidad por
  raza que se puede calibrar (sección 12). A cambio, el conjunto de razas queda fijo: sumar una raza
  obliga a reentrenar, y una raza que no vio la asigna igual a alguna de las 70.
- **Cuándo conviene cada una:** la búsqueda cuando el catálogo cambia seguido o se quiere mostrar
  ejemplos parecidos; el clasificador cuando las clases son fijas y se necesita una respuesta rápida con
  una confianza calibrada. La app usa las dos con el mismo modelo, cada una en su pestaña.

**Trade-offs.** El Modelo A está en el punto medio: la mitad de parámetros y de latencia que el baseline,
la mejor accuracy, y 43 MB que entran sin problema en el contenedor. El Modelo B solo convendría en un
dispositivo muy limitado (4 MB, 12 ms con ONNX), aceptando bastante más error. ONNX mejora la latencia de
los dos sin perder precisión; no se usa por defecto para no sumar otro formato a la entrega (D-09).

## 14. Descargar los checkpoints

Descargalos y colocalos en `models/` de tu entorno local para que la aplicación los use.
Recordá publicarlos en un link público de solo lectura y ponerlo en el README y en el PR.

In [ ]:
!ls -lh "{settings.model_path}"

try:
    from google.colab import files

    for name in (settings.resnet18_model_name, settings.cnn_custom_model_name):
        files.download(str(settings.model_path / name))
        files.download(str(settings.model_path / (Path(name).stem + "_history.json")))
except ImportError:
    print("Fuera de Colab: los checkpoints quedan en", settings.model_path)